# Analysis

In [1]:
import pandas as pd
from pathlib import Path
from urllib.parse import urldefrag, urlparse
from bs4 import BeautifulSoup
import re
from collections import Counter

In [2]:
DATA_ROOT = Path("data")
PAGES_ROOT = DATA_ROOT / "pages"
INDEX_PATH = DATA_ROOT / "file_index.csv"

In [3]:
index_df = pd.read_csv(INDEX_PATH)
index_df.head()

,urlhash,url,status
0,94a7a3dcd1da678d9dc47acf0ace402f25adddec252602...,https://www.cs.uci.edu,200
1,6dd42e6f759f153bb8b5ad1bbb7f2f67ba29767b0c99ac...,https://www.informatics.uci.edu,200
2,6d63139b0f6ff118e8268722d61003b6d1b135781a3ada...,https://www.stat.uci.edu,200
3,9a605262c4eff2499f6d2021ad392887a92717d434fb5c...,https://www.ics.uci.edu,200
4,fda6352c048c2662f4dc07de3c48faa7322f2bddc03da7...,https://ics.uci.edu,200


## How many unique pages did you find

In [4]:
unique_urls = index_df["url"].apply(lambda x: urldefrag(x)[0]).unique()
unique_urls.shape

(5476,)

## What is the longest page in terms of the number of words?

In [5]:
def clean_page(page_content: str) -> str:
    soup = BeautifulSoup(page_content, "html.parser")

    for code_block in soup(["script", "style", "code", "pre"]):
        code_block.decompose()

    return soup.get_text(separator=" ", strip=True)

def load_page(urlhash: str) -> str:
    with open(PAGES_ROOT / urlhash, "r") as f:
        return f.read()

index_df["page_content"] = index_df["urlhash"].apply(load_page).apply(clean_page)

/var/folders/8v/3hj81gys6dgdw401b74kwjrr0000gn/T/ipykernel_16238/1010472476.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(page_content, "html.parser")


In [6]:
max_index = index_df["page_content"].apply(str.split).apply(len).argmax()
index_df.iloc[max_index]["url"]

'https://cml.ics.uci.edu/category/aiml'

## What are the 50 most common words?

In [7]:
stopwords = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and",
    "any", "are", "aren't", "as", "at", "be", "because", "been", "before",
    "being", "below", "between", "both", "but", "by", "can't", "cannot", "could",
    "couldn't", "did", "didn't", "do", "does", "doesn't", "doing", "don't",
    "down", "during", "each", "few", "for", "from", "further", "had", "hadn't",
    "has", "hasn't", "have", "haven't", "having", "he", "he'd", "he'll", "he's",
    "her", "here", "here's", "hers", "herself", "him", "himself", "his", "how",
    "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into", "is",
    "isn't", "it", "it's", "its", "itself", "let's", "me", "more", "most",
    "mustn't", "my", "myself", "no", "nor", "not", "of", "off", "on", "once",
    "only", "or", "other", "ought", "our", "ours", "ourselves", "out", "over",
    "own", "same", "shan't", "she", "she'd", "she'll", "she's", "should",
    "shouldn't", "so", "some", "such", "than", "that", "that's", "the",
    "their", "theirs", "them", "themselves", "then", "there", "there's",
    "these", "they", "they'd", "they'll", "they're", "they've", "this",
    "those", "through", "to", "too", "under", "until", "up", "very", "was",
    "wasn't", "we", "we'd", "we'll", "we're", "we've", "were", "weren't",
    "what", "what's", "when", "when's", "where", "where's", "which", "while",
    "who", "who's", "whom", "why", "why's", "with", "won't", "would",
    "wouldn't", "you", "you'd", "you'll", "you're", "you've", "your", "yours",
    "yourself", "yourselves"
}

In [15]:
word_freq = Counter()
for page_content in index_df["page_content"]:
    page_content = re.sub(r"[^a-zA-Z0-9']", " ", page_content.lower())
    for word in page_content.split():
        word_freq[word.strip()] += 1

for stopword in stopwords:
    word_freq.pop(stopword, None)

pd.DataFrame(word_freq.most_common(50), columns=["Word", "Count"])

,Word,Count
0,ramesh,14102
1,research,13353
2,jain,8772
3,computer,8682
4,current,8533
5,students,7926
6,data,7926
7,information,7839
8,events,6799
9,university,6642


# How many subdomains did you find in the uci.edu domain?

In [13]:
domain_freq = Counter()
for url in unique_urls:
    if parsed := urlparse(url):
        domain_freq[parsed.hostname] += 1

pd.DataFrame(sorted(domain_freq.items(), key=lambda x: x[0]), columns=["Hostname", "Page Count"])

,Hostname,Page Count
0,accessibility.ics.uci.edu,1
1,acoi.ics.uci.edu,111
2,archive.ics.uci.edu,3
3,asterix.ics.uci.edu,7
4,asterixdb.ics.uci.edu,2
...,...,...
91,www.informatics.ics.uci.edu,1
92,www.informatics.uci.edu,15
93,www.physics.uci.edu,84
94,www.stat.uci.edu,11
